In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.datasets import CocoDetection
import torchvision.transforms as T
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

# ================================
# 1️⃣ Dataset Wrapper
# ================================

class CocoDetectionForFRCNN(CocoDetection):
    def __init__(self, root, annFile, transforms=None):
        super().__init__(root, annFile)
        self.transforms = transforms

    def __getitem__(self, idx):
        img, anno = super().__getitem__(idx)
        if self.transforms is not None:
            img = self.transforms(img)

        boxes, labels, areas, iscrowd = [], [], [], []

        for obj in anno:
            xmin, ymin, w, h = obj["bbox"]
            xmax = xmin + w
            ymax = ymin + h
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(obj["category_id"])
            areas.append(w * h)
            iscrowd.append(obj.get("iscrowd", 0))

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
            "area": torch.as_tensor(areas, dtype=torch.float32),
            "iscrowd": torch.as_tensor(iscrowd, dtype=torch.uint8),
        }

        return img, target

# ================================
# 2️⃣ Transform
# ================================

def get_transform(train):
    transforms = []
    transforms.append(T.ToTensor())  # PIL → Tensor
    return T.Compose(transforms)

# ================================
# 3️⃣ Model Definition
# ================================

def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# ================================
# 4️⃣ Visualization
# ================================

def visualize_predictions(img_tensor, preds, threshold=0.5):
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img)

    boxes = preds["boxes"].cpu().numpy()
    scores = preds["scores"].cpu().numpy()
    labels = preds["labels"].cpu().numpy()

    for i in range(len(boxes)):
        if scores[i] >= threshold:
            xmin, ymin, xmax, ymax = boxes[i]
            rect = patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                linewidth=2, edgecolor='r', facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(xmin, ymin - 5, f"{labels[i]}: {scores[i]:.2f}",
                    color="yellow", fontsize=10, backgroundcolor="black")
    plt.show()

# ================================
# 5️⃣ Training Loop with Evaluation
# ================================

def train_model(model, dataloader, val_loader, device, num_epochs=3, lr=0.005):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.0005)
    best_map = 0.0
    map_metric = MeanAveragePrecision()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for images, targets in dataloader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            running_loss += losses.item()

        avg_loss = running_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f}")

        # Evaluation
        model.eval()
        map_metric.reset()
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                preds = model(images)
                map_metric.update(preds, targets)
        metrics = map_metric.compute()
        print(f"Validation mAP: {metrics['map']:.4f}")

        # Save best checkpoint
        if metrics['map'] > best_map:
            best_map = metrics['map']
            torch.save(model.state_dict(), "best_frcnn_coco.pth")
            print(f"✅ Saved new best model (mAP={best_map:.4f})")

    print("Training complete.")

# ================================
# 6️⃣ Main Execution
# ================================

if __name__ == "__main__":
    # Paths to your COCO dataset
    train_root = "/Users/pubalimazumder/datasets/coco/train2017"
    train_ann = "/Users/pubalimazumder/datasets/coco/annotations/instances_train2017.json"
    val_root = "/Users/pubalimazumder/datasets/coco/val2017"
    val_ann = "/Users/pubalimazumder/datasets/coco/annotations/instances_val2017.json"

    # Dataset and DataLoader
    train_dataset = CocoDetectionForFRCNN(train_root, train_ann, transforms=get_transform(train=True))
    val_dataset = CocoDetectionForFRCNN(val_root, val_ann, transforms=get_transform(train=False))

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

    # Model setup
    num_classes = 91  # COCO (80 categories + background)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_model(num_classes).to(device)

    # Train and evaluate
    train_model(model, train_loader, val_loader, device, num_epochs=2)

    # Visualize one prediction
    img, _ = val_dataset[0]
    model.eval()
    with torch.no_grad():
        pred = model([img.to(device)])[0]
    visualize_predictions(img, pred, threshold=0.5)






loading annotations into memory...
Done (t=6.67s)
creating index...
index created!
loading annotations into memory...
Done (t=0.19s)
creating index...
index created!


ModuleNotFoundError: `MAP` metric requires that `pycocotools` or `faster-coco-eval` installed. Please install with `pip install pycocotools` or `pip install faster-coco-eval` or `pip install torchmetrics[detection]`.